# Notebook 04: PEFT Transfer Training — HeartBERT, ECG-PT, HuBERT-ECG

Three pretrained ECG foundation models fine-tuned on the PTB-XL 5-class diagnostic
classification task using **parameter-efficient fine-tuning (PEFT)** only.
No full fine-tuning is performed.

| Model | Architecture | HuggingFace ID | Input |
|---|---|---|---|
| HeartBERT | RoBERTa encoder | `Bayesiano/HeartBERT` | Lead II → quantised text |
| ECG-PT | GPT-2 decoder | `Tconnector/ecg-pt` | Lead II → patch tokens |
| HuBERT-ECG | HuBERT encoder | `Edoardo-BS/hubert-ecg-base` | 12-lead float tensor |

Each model is trained twice:
- **LoRA r=8** — low-rank adapter in attention Q/V projections (~0.8–1% of params)
- **DoRA r=8** — weight-decomposed LoRA; separates magnitude and direction updates

Results saved to `results/<experiment_name>/`.

In [1]:
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '0'   # allow HF download on first run
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import torch
import wfdb

from src.utils.config import CFG
from src.preprocessing.label_utils import load_all_labels, SUPERCLASSES

DATA_PATH    = CFG['data']['path']
RESULTS_PATH = CFG['paths']['results']
HUBERT_SIZE  = CFG['model']['hubert_size']   # "base"
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device       : {device}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Results dir  : {RESULTS_PATH}')
print(f'HuBERT size  : {HUBERT_SIZE}')

Device       : cuda
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM         : 8.6 GB
Results dir  : D:\GitHub\biosignal-xai\results/
HuBERT size  : base


## 2. Data

PTB-XL folds: 1–8 train, 9 val, 10 test (never touched here).

**Two loading paths:**
- `X_lead` / `y_*` — Lead II (index 1) as numpy arrays for HeartBERT and ECG-PT.
  Loaded upfront because HuggingFace tokenisers cannot run inside DataLoader workers.
- `ECGDatasetFull` — 12-lead on-the-fly loader for HuBERT-ECG (avoids ~800 MB preload).

In [2]:
from src.preprocessing.dataset_full import ECGDatasetFull

Y = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')

train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]

# ── Single-lead numpy arrays (HeartBERT / ECG-PT) ─────────────────────────
LEAD_IDX = 1   # Lead II

def _load_lead(df, data_path, lead_idx=LEAD_IDX):
    X, y = [], []
    for i, (_, row) in enumerate(df.iterrows()):
        sig, _ = wfdb.rdsamp(data_path + row['filename_lr'])
        X.append(sig[:, lead_idx].astype(np.float32))
        y.append(np.array(row['label_vec'], dtype=np.float32))
        if (i + 1) % 2000 == 0:
            print(f"  Loaded {i+1}/{len(df)} records...")
    return np.stack(X), np.stack(y)

print("Loading single-lead arrays...")
X_train_lead, y_train = _load_lead(train_df, DATA_PATH)
X_val_lead,   y_val   = _load_lead(val_df,   DATA_PATH)

# ── Full 12-lead datasets (HuBERT-ECG) ────────────────────────────────────
train_ds_full = ECGDatasetFull(train_df, DATA_PATH)
val_ds_full   = ECGDatasetFull(val_df,   DATA_PATH)

print(f'\nSingle-lead arrays:')
print(f'  X_train_lead : {X_train_lead.shape}  y_train : {y_train.shape}')
print(f'  X_val_lead   : {X_val_lead.shape}  y_val   : {y_val.shape}')
print(f'\nFull-lead datasets:')
print(f'  train_ds_full : {len(train_ds_full):,} records')
print(f'  val_ds_full   : {len(val_ds_full):,} records')

Records with valid labels: 21375
Class distribution:
  NORM: 9514 (44.5%)
  MI: 5469 (25.6%)
  STTC: 5108 (23.9%)
  CD: 4898 (22.9%)
  HYP: 2649 (12.4%)
Loading single-lead arrays...
  Loaded 2000/17073 records...
  Loaded 4000/17073 records...
  Loaded 6000/17073 records...
  Loaded 8000/17073 records...
  Loaded 10000/17073 records...
  Loaded 12000/17073 records...
  Loaded 14000/17073 records...
  Loaded 16000/17073 records...
  Loaded 2000/2145 records...

Single-lead arrays:
  X_train_lead : (17073, 1000)  y_train : (17073, 5)
  X_val_lead   : (2145, 1000)  y_val   : (2145, 5)

Full-lead datasets:
  train_ds_full : 17,073 records
  val_ds_full   : 2,145 records


## 3. HeartBERT

RoBERTa encoder pretrained on ECG-as-text. Input ECG signals are
quantised into 20-bin letter strings before tokenisation.

Each run:
1. Fresh `HeartBERTClassifier` instance loaded from `Bayesiano/HeartBERT`
2. PEFT adapters attached (LoRA or DoRA)
3. Sanity check: output shape `(2, 5)`, trainable < 5 %
4. `fit()` — trains with AdamW, BCEWithLogitsLoss, saves best checkpoint

In [3]:
from src.models.heartbert import HeartBERTClassifier

_probe = HeartBERTClassifier(num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)

# Shape check: single dummy signal
_dummy_X = np.random.randn(2, 1000).astype(np.float32)
_dummy_p = _probe.predict(_dummy_X)
assert _dummy_p.shape == (2, 5), f"Shape error: {_dummy_p.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 1000) -> {_dummy_p.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe

Bayesiano/HeartBERT not available — loading roberta-base (same architecture).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 889,349 || all params: 125,538,826 || trainable%: 0.7084
Shape OK : (2, 1000) -> (2, 5)
Params   : 889,349 / 125,538,826 = 0.7%


In [4]:
from src.models.heartbert import HeartBERTClassifier

model_hb_lora = HeartBERTClassifier(num_labels=5)
model_hb_lora.load()
model_hb_lora.apply_peft(use_dora=False)

auc_hb_lora, hist_hb_lora = model_hb_lora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'heartbert_lora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_hb_lora
import gc; gc.collect()
torch.cuda.empty_cache()

Bayesiano/HeartBERT not available — loading roberta-base (same architecture).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 889,349 || all params: 125,538,826 || trainable%: 0.7084
  Tokenising signals (this takes ~1 min for 17k records)...

 Experiment : heartbert_lora_r8
 Device     : cuda
 Trainable  : 889,349  (0.7%)

Epoch 01/15  train=0.4788  val=0.4574  AUC=0.7662
  * Best saved — AUC 0.7662
Epoch 02/15  train=0.4383  val=0.4356  AUC=0.7820
  * Best saved — AUC 0.7820
Epoch 03/15  train=0.4261  val=0.4333  AUC=0.7860
  * Best saved — AUC 0.7860
Epoch 04/15  train=0.4191  val=0.4293  AUC=0.7904
  * Best saved — AUC 0.7904
Epoch 05/15  train=0.4122  val=0.4219  AUC=0.7967
  * Best saved — AUC 0.7967
Epoch 06/15  train=0.4063  val=0.4184  AUC=0.8036
  * Best saved — AUC 0.8036
Epoch 07/15  train=0.3993  val=0.4223  AUC=0.8050
  * Best saved — AUC 0.8050
Epoch 08/15  train=0.3935  val=0.4115  AUC=0.8111
  * Best saved — AUC 0.8111
Epoch 09/15  train=0.3877  val=0.4393  AUC=0.8053
Epoch 10/15  train=0.3841  val=0.4111  AUC=0.8152
  * Best saved — AUC 0.8152
Epoch 11/15  train=0.3

In [5]:
from src.models.heartbert import HeartBERTClassifier

model_hb_dora = HeartBERTClassifier(num_labels=5)
model_hb_dora.load()
model_hb_dora.apply_peft(use_dora=True)

auc_hb_dora, hist_hb_dora = model_hb_dora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'heartbert_dora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_hb_dora
import gc; gc.collect()
torch.cuda.empty_cache()

Bayesiano/HeartBERT not available — loading roberta-base (same architecture).


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 907,781 || all params: 125,557,258 || trainable%: 0.7230
  Tokenising signals (this takes ~1 min for 17k records)...

 Experiment : heartbert_dora_r8
 Device     : cuda
 Trainable  : 907,781  (0.7%)

Epoch 01/15  train=0.4767  val=0.4742  AUC=0.7714
  * Best saved — AUC 0.7714
Epoch 02/15  train=0.4359  val=0.4355  AUC=0.7849
  * Best saved — AUC 0.7849
Epoch 03/15  train=0.4235  val=0.4343  AUC=0.7864
  * Best saved — AUC 0.7864
Epoch 04/15  train=0.4178  val=0.4218  AUC=0.7926
  * Best saved — AUC 0.7926
Epoch 05/15  train=0.4107  val=0.4220  AUC=0.7934
  * Best saved — AUC 0.7934
Epoch 06/15  train=0.4052  val=0.4228  AUC=0.7957
  * Best saved — AUC 0.7957
Epoch 07/15  train=0.3987  val=0.4310  AUC=0.7987
  * Best saved — AUC 0.7987
Epoch 08/15  train=0.3934  val=0.4091  AUC=0.8060
  * Best saved — AUC 0.8060
Epoch 09/15  train=0.3882  val=0.4172  AUC=0.8045
Epoch 10/15  train=0.3827  val=0.4068  AUC=0.8127
  * Best saved — AUC 0.8127
Epoch 11/15  train=0.3

## 4. ECG-PT (supervised)

GPT-2 based decoder pretrained on ECG time-series. Originally unsupervised
(reconstruction loss). Adapted here for supervised classification by replacing
the causal-LM head with a sequence-classification head (last token → linear → 5).

Input signals are split into 36-sample patches and quantised to token IDs.
Falls back to plain GPT-2 architecture if the `Tconnector/ecg-pt` checkpoint
is not yet publicly available.

In [6]:
from src.models.ecgpt import ECGPTClassifier

_probe = ECGPTClassifier(num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)

_dummy_p = _probe.predict(np.random.randn(2, 1000).astype(np.float32))
assert _dummy_p.shape == (2, 5), f"Shape error: {_dummy_p.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 1000) -> {_dummy_p.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe

Tconnector/ecg-pt not available — loading GPT-2 (same architecture).


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 298,752 || all params: 124,742,400 || trainable%: 0.2395
Shape OK : (2, 1000) -> (2, 5)
Params   : 298,752 / 124,742,400 = 0.2%


In [7]:
from src.models.ecgpt import ECGPTClassifier

model_ecgpt_lora = ECGPTClassifier(num_labels=5)
model_ecgpt_lora.load()
model_ecgpt_lora.apply_peft(use_dora=False)

auc_ecgpt_lora, hist_ecgpt_lora = model_ecgpt_lora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'ecgpt_lora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_ecgpt_lora
import gc; gc.collect()
torch.cuda.empty_cache()

Tconnector/ecg-pt not available — loading GPT-2 (same architecture).


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 298,752 || all params: 124,742,400 || trainable%: 0.2395
  Tokenising signals...

 Experiment : ecgpt_lora_r8
 Device     : cuda
 Trainable  : 298,752  (0.2%)

Epoch 01/15  train=0.5623  val=0.5104  AUC=0.6262
  * Best saved — AUC 0.6262
Epoch 02/15  train=0.5154  val=0.5197  AUC=0.6290
  * Best saved — AUC 0.6290
Epoch 03/15  train=0.5110  val=0.5056  AUC=0.6330
  * Best saved — AUC 0.6330
Epoch 04/15  train=0.5081  val=0.5040  AUC=0.6339
  * Best saved — AUC 0.6339
Epoch 05/15  train=0.5070  val=0.5084  AUC=0.6318
Epoch 06/15  train=0.5058  val=0.5068  AUC=0.6338
Epoch 07/15  train=0.5045  val=0.5149  AUC=0.6325
Epoch 08/15  train=0.5030  val=0.5070  AUC=0.6349
  * Best saved — AUC 0.6349
Epoch 09/15  train=0.5024  val=0.5045  AUC=0.6324
Epoch 10/15  train=0.5017  val=0.5057  AUC=0.6362
  * Best saved — AUC 0.6362
Epoch 11/15  train=0.5003  val=0.5056  AUC=0.6343
Epoch 12/15  train=0.4985  val=0.5057  AUC=0.6339
Epoch 13/15  train=0.4978  val=0.5074  AUC=0.6

In [8]:
from src.models.ecgpt import ECGPTClassifier

model_ecgpt_dora = ECGPTClassifier(num_labels=5)
model_ecgpt_dora.load()
model_ecgpt_dora.apply_peft(use_dora=True)

auc_ecgpt_dora, hist_ecgpt_dora = model_ecgpt_dora.fit(
    X_train_lead, y_train,
    X_val_lead,   y_val,
    experiment_name = 'ecgpt_dora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_peft'],
    save_dir        = RESULTS_PATH,
)
del model_ecgpt_dora
import gc; gc.collect()
torch.cuda.empty_cache()

Tconnector/ecg-pt not available — loading GPT-2 (same architecture).


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Done.
trainable params: 326,400 || all params: 124,770,048 || trainable%: 0.2616
  Tokenising signals...

 Experiment : ecgpt_dora_r8
 Device     : cuda
 Trainable  : 326,400  (0.3%)

Epoch 01/15  train=0.6276  val=0.5187  AUC=0.6179
  * Best saved — AUC 0.6179
Epoch 02/15  train=0.5192  val=0.5058  AUC=0.6321
  * Best saved — AUC 0.6321
Epoch 03/15  train=0.5128  val=0.5078  AUC=0.6356
  * Best saved — AUC 0.6356
Epoch 04/15  train=0.5093  val=0.5076  AUC=0.6370
  * Best saved — AUC 0.6370
Epoch 05/15  train=0.5073  val=0.5052  AUC=0.6356
Epoch 06/15  train=0.5057  val=0.5036  AUC=0.6369
Epoch 07/15  train=0.5049  val=0.5140  AUC=0.6372
  * Best saved — AUC 0.6372
Epoch 08/15  train=0.5041  val=0.5081  AUC=0.6327
Epoch 09/15  train=0.5034  val=0.5042  AUC=0.6317
Epoch 10/15  train=0.5021  val=0.5038  AUC=0.6328
Epoch 11/15  train=0.5008  val=0.5124  AUC=0.6301
Epoch 12/15  train=0.4995  val=0.5088  AUC=0.6289
Epoch 13/15  train=0.4982  val=0.5134  AUC=0.6274
Epoch 14/15  train=0.497

## 5. HuBERT-ECG

Foundation model pretrained on 9.1 M 12-lead ECGs with masked self-supervised
prediction across 164 conditions. Unlike HeartBERT and ECG-PT, it processes
all 12 leads simultaneously and is loaded via `ECGDatasetFull` + `run_peft_experiment()`
for memory-efficient streaming.

LoRA adapters target `q_proj` and `v_proj` in the HuBERT transformer blocks.

In [3]:
from src.models.hubert_ecg import HuBERTECGClassifier

_probe = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
_probe.load()
_probe.apply_peft(use_dora=False)
_probe.to(device)

_dummy_x = torch.randn(2, 12, 1000, device=device)
with torch.no_grad():
    _out = _probe(_dummy_x)
assert _out.shape == (2, 5), f"Shape error: {_out.shape}"

_params = _probe.count_parameters()
assert _params['trainable'] / _params['total'] < 0.05, \
    f"Trainable fraction {_params['trainable']/_params['total']:.1%} exceeds 5%"

print(f"Shape OK : (2, 12, 1000) -> {_out.shape}")
print(f"Params   : {_params['trainable']:,} / {_params['total']:,} = {_params['percentage']}")
del _probe, _dummy_x, _out
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 294,912 || all params: 93,424,005 || trainable%: 0.3157
Shape OK : (2, 12, 1000) -> torch.Size([2, 5])
Params   : 294,912 / 93,424,005 = 0.3%


In [4]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.training.train_peft import run_peft_experiment

model_hubert_lora = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model_hubert_lora.load()
model_hubert_lora.apply_peft(use_dora=False)
model_hubert_lora.to(device)

auc_hubert_lora, hist_hubert_lora, _ = run_peft_experiment(
    model_hubert_lora, train_ds_full, val_ds_full,
    experiment_name = 'hubert_ecg_lora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS_PATH,
    num_workers     = CFG['training']['num_workers'],
)
del model_hubert_lora
import gc; gc.collect()
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 294,912 || all params: 93,424,005 || trainable%: 0.3157

 Experiment : hubert_ecg_lora_r8
 Device     : cuda:0
 Trainable  : 294,912  (0.3%)

Epoch 01/15  lr=9.89e-05
  train_loss=0.4138  val_loss=0.4101
  AUC (macro): 0.8270
  Per-class AUC:
    NORM : 0.892  #################
    MI   : 0.829  ################
    STTC : 0.891  #################
    CD   : 0.805  ################
    HYP  : 0.719  ##############
Adapter saved → D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r8\best_adapter
Saved → D:\GitHub\biosignal-xai\results/hubert_ecg_lora_r8\best_adapter
  * Best saved — AUC 0.8270
Epoch 02/15  lr=9.57e-05
  train_loss=0.3618  val_loss=0.4013
  AUC (macro): 0.8454
  Per-class AUC:
    NORM : 0.904  ##################
    MI   : 0.861  #################
    STTC : 0.903  ##################
    CD   : 0.826  ################
    HYP  : 0.733  ##############
Adapter saved → D:\

In [5]:
from src.models.hubert_ecg import HuBERTECGClassifier
from src.training.train_peft import run_peft_experiment

model_hubert_dora = HuBERTECGClassifier(size=HUBERT_SIZE, num_labels=5)
model_hubert_dora.load()
model_hubert_dora.apply_peft(use_dora=True)
model_hubert_dora.to(device)

auc_hubert_dora, hist_hubert_dora, _ = run_peft_experiment(
    model_hubert_dora, train_ds_full, val_ds_full,
    experiment_name = 'hubert_ecg_dora_r8',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_pretrained'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = RESULTS_PATH,
    num_workers     = CFG['training']['num_workers'],
)
del model_hubert_dora
import gc; gc.collect()
torch.cuda.empty_cache()

Loading HuBERT-ECG (base) from Edoardo-BS/hubert-ecg-base ...
  Done. Hidden dim: 768
trainable params: 313,344 || all params: 93,442,437 || trainable%: 0.3353

 Experiment : hubert_ecg_dora_r8
 Device     : cuda:0
 Trainable  : 313,344  (0.3%)

Epoch 01/15  lr=9.89e-05
  train_loss=0.4123  val_loss=0.3783
  AUC (macro): 0.8377
  Per-class AUC:
    NORM : 0.902  ##################
    MI   : 0.840  ################
    STTC : 0.900  #################
    CD   : 0.817  ################
    HYP  : 0.730  ##############
Adapter saved → D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r8\best_adapter
Saved → D:\GitHub\biosignal-xai\results/hubert_ecg_dora_r8\best_adapter
  * Best saved — AUC 0.8377
Epoch 02/15  lr=9.57e-05
  train_loss=0.3584  val_loss=0.3605
  AUC (macro): 0.8562
  Per-class AUC:
    NORM : 0.916  ##################
    MI   : 0.882  #################
    STTC : 0.908  ##################
    CD   : 0.832  ################
    HYP  : 0.742  ##############
Adapter saved → D:

## 6. Summary

Validation macro-AUC across all six experiments. Trainable parameters and checkpoint
sizes reported from profiling.json files.

In [7]:
import json
from pathlib import Path

In [11]:
experiments = [
    ('heartbert_lora_r8',   auc_hb_lora),
    ('heartbert_dora_r8',   auc_hb_dora),
    ('ecgpt_lora_r8',       auc_ecgpt_lora),
    ('ecgpt_dora_r8',       auc_ecgpt_dora),
    # ('hubert_ecg_lora_r8',  auc_hubert_lora),
    # ('hubert_ecg_dora_r8',  auc_hubert_dora),
]

print(f"{'Experiment':<28}  {'Val AUC':>8}  {'Trainable':>12}  {'Ckpt MB':>8}")
print("-" * 62)

for exp_name, best_auc in experiments:
    prof_path = Path(RESULTS_PATH) / exp_name / 'profiling.json'
    if prof_path.exists():
        prof = json.loads(prof_path.read_text())
        trainable  = prof['trainable_params']
        ckpt_mb    = prof['checkpoint_size_mb']
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {trainable:>12,}  {ckpt_mb:>7.1f} MB")
    else:
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {'N/A':>12}  {'N/A':>8}")

print(f"\nBaseline FCN-Wang AUC (test set): 0.9225  (notebook 03)")
print(f"Full comparison plots → notebook 05")

Experiment                     Val AUC     Trainable   Ckpt MB
--------------------------------------------------------------
heartbert_lora_r8               0.8258       889,349      3.6 MB
heartbert_dora_r8               0.8199       907,781      3.6 MB
ecgpt_lora_r8                   0.6362       298,752      1.2 MB
ecgpt_dora_r8                   0.6372       326,400      1.3 MB

Baseline FCN-Wang AUC (test set): 0.9225  (notebook 03)
Full comparison plots → notebook 05


In [8]:
hubert_exp = [
    ('hubert_ecg_lora_r8',  auc_hubert_lora),
    ('hubert_ecg_dora_r8',  auc_hubert_dora),
]

print(f"{'Experiment':<28}  {'Val AUC':>8}  {'Trainable':>12}  {'Ckpt MB':>8}")
print("-" * 62)

for exp_name, best_auc in hubert_exp:
    prof_path = Path(RESULTS_PATH) / exp_name / 'profiling.json'
    if prof_path.exists():
        prof = json.loads(prof_path.read_text())
        trainable  = prof['trainable_params']
        ckpt_mb    = prof['checkpoint_size_mb']
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {trainable:>12,}  {ckpt_mb:>7.1f} MB")
    else:
        print(f"{exp_name:<28}  {best_auc:>8.4f}  {'N/A':>12}  {'N/A':>8}")

print(f"\nBaseline FCN-Wang AUC (test set): 0.9225  (notebook 03)")
print(f"Full comparison plots → notebook 05")

Experiment                     Val AUC     Trainable   Ckpt MB
--------------------------------------------------------------
hubert_ecg_lora_r8              0.8687       294,912      1.2 MB
hubert_ecg_dora_r8              0.8680       313,344      1.3 MB

Baseline FCN-Wang AUC (test set): 0.9225  (notebook 03)
Full comparison plots → notebook 05
